# Transformación — `matriz_nut_100g`, `category_stats` y D2 (NutriMatch)

**Snapshot analizado:** `off_csv_20260919` (16.851 productos de México).

**Fecha de ejecución:** 2026-09-20.

**Objetivo (paso 6 del plan):** construir la matriz de nutrición saneada, resolver la categoría de referencia de cada producto, calcular los percentiles de D1 dentro de esa categoría, construir `category_stats` y calcular el subpuntaje D2. Todo con trazabilidad al `snapshot_id`.

**Este notebook implementa seis decisiones cerradas el 2026-09-20** (`AGENTS.md`, A15-A20), resultado de `notebooks/02_eda_universo_mexico.ipynb`:

- **A16** (categoría de referencia): retroceso ascendente, umbral mínimo 30 productos.
- **A17** (D2): NOVA combinado con `additives_n` dentro de cada grupo.
- **A18** (saneamiento): valores fuera de rango físico, flag + exclusión de percentiles.
- **A20** (fibra): incluida en el núcleo global (core8).

**Lo que este notebook NO hace** (queda para el paso 7, modelo de recomendación): aplicar el signo de D1 según el objetivo del usuario, calcular D3 (preferencias), aplicar la regla de cobertura `cov` con pesos del usuario, ni los filtros duros de alergia y dieta (A15). Los percentiles que se calculan aquí son **agnósticos del usuario**: 0 es el valor más bajo del grupo de referencia, 100 el más alto, sin decidir todavía si eso es deseable.

**Todo el código de transformación vive en `src/nutrimatch/engine`** (no en este notebook): aquí solo se invoca, se verifica con datos reales y se documenta. El motor tiene 17 pruebas unitarias en `tests/` que pasan antes de ejecutar esta transformación.

**Aviso:** sin evaluación médica. Este notebook produce features derivadas de datos de producto, no recomendaciones de compra.

---


In [1]:
"""Configuración e ingesta del snapshot.

Se cargan a pandas solo las columnas que necesita el motor: el snapshot completo
tiene 211 columnas, pero la transformación de esta etapa usa 12. El resto queda
en el Parquet original para cuando se necesiten (búsqueda, ficha de producto).
"""

import json
from pathlib import Path

import duckdb
import pandas as pd
import plotly.express as px

from nutrimatch.engine.constants import CORE8_NUTRIENTES, MINIMO_PEERS_PERCENTIL, TAMANO_MINIMO_CATEGORIA
from nutrimatch.engine.nutrition_percentile import calcular_percentiles_por_categoria, construir_category_stats
from nutrimatch.engine.processing_score import calcular_d2, calibrar_tope_aditivos
from nutrimatch.engine.reference_category import resolver_categoria_referencia
from nutrimatch.engine.sanitize import (
    corregir_escala_sal_por_sodio,
    flag_suma_macros_excede_100,
    sanear_nucleo_nutricional,
)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SNAPSHOT_ID = "off_csv_20260919"
PARQUET_PATH = REPO_ROOT / "datos" / "procesados" / "off_mexico_20260919.parquet"
META_PATH = REPO_ROOT / "datos" / "snapshots" / SNAPSHOT_ID / "_metadata.json"

metadata = json.loads(META_PATH.read_text(encoding="utf-8"))

con = duckdb.connect()
columnas_necesarias = [
    *CORE8_NUTRIENTES,
    "code",
    "categories_tags",
    "nova_group",
    "additives_n",
    "sodium_100g",  # fuera del núcleo core8; se usa solo para la corrección de escala de sal
]
seleccion_sql = ", ".join(f'"{c}"' for c in columnas_necesarias)
df = con.execute(f"SELECT {seleccion_sql} FROM '{PARQUET_PATH}'").df()

N = len(df)
print(f"snapshot_id: {SNAPSHOT_ID}")
print(f"Productos cargados: N = {N:,}")
print(f"Umbral mínimo de categoría (A16): {TAMANO_MINIMO_CATEGORIA}")
print(f"Mínimo de pares para percentil (salvaguarda añadida en transformación): {MINIMO_PEERS_PERCENTIL}")

snapshot_id: off_csv_20260919
Productos cargados: N = 16,851
Umbral mínimo de categoría (A16): 30
Mínimo de pares para percentil (salvaguarda añadida en transformación): 5


## 1. Saneamiento del núcleo nutricional (decisión A18)

`src/nutrimatch/engine/sanitize.py`. Por cada uno de los 8 nutrientes núcleo (core8, decisión A20), se separan tres columnas: el valor bruto (dato original, solo convertido a número), una bandera de fuera de rango físico, y el valor saneado (el bruto, pero NaN si está fuera de rango). Solo el valor saneado alimenta los percentiles y `category_stats` de las secciones siguientes.


In [2]:
saneado = sanear_nucleo_nutricional(df)
flag_macros = flag_suma_macros_excede_100(saneado)

filas = []
for nutriente in CORE8_NUTRIENTES:
    con_dato = saneado[f"{nutriente}_bruto"].notna().sum()
    fuera_de_rango = saneado[f"{nutriente}_flag_fuera_de_rango"].sum()
    filas.append({
        "nutriente": nutriente,
        "con_dato_bruto": con_dato,
        "pct_con_dato": round(con_dato * 100 / N, 1),
        "fuera_de_rango": int(fuera_de_rango),
        "saneado_valido": saneado[f"{nutriente}_saneado"].notna().sum(),
    })

tabla_saneamiento = pd.DataFrame(filas)
print(f"Fila con suma grasa+carbohidratos+proteína > 100,5 g: {flag_macros.sum()} ({flag_macros.sum()*100/N:.2f}%)")
tabla_saneamiento

Fila con suma grasa+carbohidratos+proteína > 100,5 g: 57 (0.34%)


,nutriente,con_dato_bruto,pct_con_dato,fuera_de_rango,saneado_valido
0,energy-kcal_100g,12241,72.6,58,12183
1,fat_100g,12024,71.4,6,12018
2,saturated-fat_100g,10491,62.3,4,10487
3,carbohydrates_100g,12060,71.6,25,12035
4,sugars_100g,10554,62.6,13,10541
5,proteins_100g,12010,71.3,5,12005
6,salt_100g,10223,60.7,56,10167
7,fiber_100g,10294,61.1,3,10291


In [3]:
# Muestra de los productos con salt_100g fuera de rango: se revisa junto con
# sodium_100g (fuera del núcleo core8, pero disponible en el snapshot) para
# distinguir un error aleatorio de uno sistemático.
con.execute(f"""
    SELECT code, product_name, salt_100g, sodium_100g
    FROM '{PARQUET_PATH}'
    WHERE try_cast(salt_100g AS DOUBLE) < 0 OR try_cast(salt_100g AS DOUBLE) > 100
    ORDER BY try_cast(salt_100g AS DOUBLE) DESC LIMIT 10
""").df()

,code,product_name,salt_100g,sodium_100g
0,7503016177955,NaN,68425,27370
1,7500326381140,Steakos carne seca,6532.5,2613
2,0009733900005,Valentina Salsa picante,5075,2030
3,7500464121523,Coco & Cacao,3750,1500
4,7501006560527,NaN,3110,1244
5,7501040003073,Salchicha Chimex para asar,2975,1190
6,7501253646951,Palomitas,2862.5,1145
7,7503053835771,NaN,2487.5,995
8,7501006559026,Act II sabor Mantequilla Extra,2166.66666666667,866.666666666667
9,7501253666300,Tortillas de harina de trigo,2090.3,836.12


### Hallazgo 1 — el saneamiento afecta a una fracción pequeña de cada nutriente; la sal revela un error sistemático de escala, no aleatorio

Verificado el 2026-09-20: los productos marcados por fuera de rango van de 3 (fibra) a 58 (energía). La bandera de suma de macros (57 productos, 0,34% del universo) es coherente con el 0,71% medido en el EDA sobre el subconjunto que tenía los tres macronutrientes a la vez.

`salt_100g` sorprende con 56 casos fuera de rango — más que saturadas, proteína o fibra —, algo que el EDA no había desglosado nutriente por nutriente. Al revisar la muestra (celda siguiente), aparece un patrón que **no es ruido**: todos los valores mantienen exactamente la razón sal ≈ 2,5 × sodio del producto correcto (por ejemplo, 68.425 / 27.370 = 2,5 exacto), solo que ambos números están inflados por el mismo factor. Es decir, el par sal-sodio es **internamente consistente**, pero la magnitud absoluta está mal — compatible con una captura en miligramos donde OFF esperaba gramos. No es una corrupción aleatoria como el caso de energía del EDA: es un patrón sistemático.

**Decisión (2026-09-20, confirmada por Paola tras la investigación de la celda siguiente):** se corrige automáticamente cuando la razón sal/sodio es ≈2,5 y el valor reescalado (÷1000) cae en rango válido — 55 de los 56 casos, ver `sanitize.corregir_escala_sal_por_sodio`. El caso atípico (razón ≈2500, un error de otra naturaleza) queda fuera de la regla y sigue marcado sin corregir, tal como se documenta en la sección "Corrección de escala de sal" más abajo.

Ningún nutriente pierde más del 0,4% de su cobertura bruta por el saneamiento.


### Corrección de escala de sal (decisión A21, confirmada 2026-09-20)

`src/nutrimatch/engine/sanitize.py::corregir_escala_sal_por_sodio`. Verifica, para cada producto con `salt_100g` fuera de rango, si su razón con `sodium_100g` es ≈2,5 (la fórmula química estándar sal = sodio × 2,5, no un ajuste específico de este dataset) y si el valor reescalado (÷1000) cae dentro del rango válido. Si ambas condiciones se cumplen, `salt_100g_saneado` deja de ser NaN y pasa a valer el reescalado; se registra en `salt_100g_flag_correccion_escala_aplicada` para que quede trazable qué productos tienen un dato "tal cual" y cuáles uno "corregido por esta regla". El caso que no cumple la razón (Queso panela, razón ≈2500) se queda fuera y sigue anulado.


In [4]:
salt_fuera_de_rango_antes = int(saneado["salt_100g_flag_fuera_de_rango"].sum())
salt_saneado_valido_antes = int(saneado["salt_100g_saneado"].notna().sum())

saneado = corregir_escala_sal_por_sodio(df, saneado)

n_corregidos = int(saneado["salt_100g_flag_correccion_escala_aplicada"].sum())
salt_saneado_valido_despues = int(saneado["salt_100g_saneado"].notna().sum())
sin_corregir_aun_fuera_de_rango = salt_fuera_de_rango_antes - n_corregidos

print(f"salt_100g fuera de rango antes de la corrección:  {salt_fuera_de_rango_antes}")
print(f"Corregidos por la regla (razón ≈2,5, resultado válido): {n_corregidos}")
print(f"Siguen fuera de rango sin corregir (no cumplen la razón): {sin_corregir_aun_fuera_de_rango}")
print(f"\nsalt_100g_saneado con dato válido: {salt_saneado_valido_antes} → {salt_saneado_valido_despues} "
      f"(+{salt_saneado_valido_despues - salt_saneado_valido_antes})")

# Verificación puntual: el caso "Steakos carne seca" (salt=6532.5, sodium=2613)
# debe terminar en 6.5325 g/100g, un valor de sal perfectamente plausible.
fila_verificacion = df[df["code"] == "7500326381140"].index
print(f"\nVerificación puntual (Steakos carne seca): salt_100g_saneado = "
      f"{saneado.loc[fila_verificacion, 'salt_100g_saneado'].iloc[0]:.4f} g/100g")

salt_100g fuera de rango antes de la corrección:  56
Corregidos por la regla (razón ≈2,5, resultado válido): 55
Siguen fuera de rango sin corregir (no cumplen la razón): 1

salt_100g_saneado con dato válido: 10167 → 10222 (+55)

Verificación puntual (Steakos carne seca): salt_100g_saneado = 6.5325 g/100g


**Resultado:** 55 de los 56 casos se corrigen y pasan a tener un valor saneado válido; el caso atípico (Queso panela) permanece anulado, como corresponde. `salt_100g` gana 55 productos con dato saneado utilizable para percentiles, sin haber "inventado" ningún valor: cada corrección se deriva del propio `sodium_100g` del producto mediante una fórmula química universal, no de un supuesto externo al dataset.


## 2. Categoría de referencia por retroceso ascendente (decisión A16)

`src/nutrimatch/engine/reference_category.py`. Para cada producto: se toma su etiqueta más específica de `categories_tags`; si el número de productos que traen esa etiqueta en cualquier posición es menor que 30, se sube un nivel y se repite. `nivel_retroceso` registra cuántas veces hubo que subir (0 = se usó la más específica).


In [5]:
categoria = resolver_categoria_referencia(df["categories_tags"], tamano_minimo=TAMANO_MINIMO_CATEGORIA)

con_categorias_tags = df["categories_tags"].notna().sum()
con_referencia = categoria["categoria_referencia"].notna().sum()
print(f"Productos con al menos una etiqueta en categories_tags: {con_categorias_tags:,} ({con_categorias_tags*100/N:.1f}%)")
print(f"Productos con categoría de referencia resuelta:         {con_referencia:,} ({con_referencia*100/N:.1f}%)")
print(f"  de los cuales, sin poder resolver ninguna (agotaron la jerarquía): "
      f"{con_categorias_tags - con_referencia:,}\n")

distribucion_retroceso = categoria["nivel_retroceso"].value_counts(dropna=False).sort_index()
distribucion_retroceso.index = distribucion_retroceso.index.map(
    lambda x: "sin resolver" if pd.isna(x) else int(x)
)
distribucion_retroceso.rename("n_productos").to_frame()

Productos con al menos una etiqueta en categories_tags: 8,728 (51.8%)
Productos con categoría de referencia resuelta:         8,101 (48.1%)
  de los cuales, sin poder resolver ninguna (agotaron la jerarquía): 627



,n_productos
nivel_retroceso,
0,4488
1,1997
2,825
3,364
4,204
5,106
6,43
7,31
8,13


In [6]:
grafico = distribucion_retroceso[distribucion_retroceso.index != "sin resolver"].reset_index()
grafico.columns = ["nivel_retroceso", "n_productos"]
fig = px.bar(
    grafico, x="nivel_retroceso", y="n_productos",
    title="Niveles de retroceso necesarios para resolver la categoría de referencia (A16)",
    labels={"nivel_retroceso": "Veces que se subió de nivel", "n_productos": "Nº de productos"},
)
fig.update_layout(xaxis=dict(dtick=1))
fig.show()

### Hallazgo 2 — la regla resuelve el 48,1% del universo, menos que el 51,8% que tiene alguna categoría

Verificado el 2026-09-20: 8.101 de 16.851 productos (48,1%) obtienen una categoría de referencia válida, frente al 51,8% (8.728) que tiene *alguna* etiqueta en `categories_tags`. La diferencia — 627 productos — son productos cuya jerarquía completa de categorías nunca alcanza 30 miembros ni siquiera en su etiqueta más genérica: la regla los deja sin categoría de referencia en vez de forzar una, que es el comportamiento correcto según A2 (sin dato es NULL, nunca una aproximación forzada), pero conviene que quede explícito que existe este costo.

De los que sí resuelven, el 55,4% (4.488) usa directamente su categoría más específica sin necesitar retroceso, y la mayoría del resto retrocede solo 1 o 2 niveles. Hay una cola larga de casos que retroceden muchas veces (hasta 21 niveles en un caso), señal de productos con jerarquías de categoría inusualmente profundas y fragmentadas en OFF; no afecta la validez de la regla, pero es una particularidad de los datos que vale la pena tener presente al leer `category_stats`: una categoría muy genérica (resultado de mucho retroceso) agrupa productos que probablemente son heterogéneos entre sí, aunque cumplan el mínimo de 30.


## 3. Percentiles de D1 y `category_stats`

`src/nutrimatch/engine/nutrition_percentile.py`. El percentil de un nutriente es NaN si el producto no tiene categoría de referencia, no tiene dato saneado para ese nutriente, o si menos de 5 productos de su categoría (salvaguarda añadida en esta transformación, ver `constants.MINIMO_PEERS_PERCENTIL`) tienen dato válido para compararlo. Es agnóstico del objetivo: 100 = valor más alto de su grupo, no "mejor".


In [7]:
percentiles = calcular_percentiles_por_categoria(
    saneado, categoria["categoria_referencia"], minimo_peers=MINIMO_PEERS_PERCENTIL
)

filas = []
for nutriente in CORE8_NUTRIENTES:
    col = f"percentil_{nutriente}"
    n_valido = percentiles[col].notna().sum()
    filas.append({"nutriente": nutriente, "con_percentil": n_valido, "pct_del_universo": round(n_valido * 100 / N, 1)})
pd.DataFrame(filas)

,nutriente,con_percentil,pct_del_universo
0,energy-kcal_100g,6975,41.4
1,fat_100g,6995,41.5
2,saturated-fat_100g,6690,39.7
3,carbohydrates_100g,6936,41.2
4,sugars_100g,6743,40.0
5,proteins_100g,6949,41.2
6,salt_100g,6734,40.0
7,fiber_100g,6549,38.9


In [8]:
category_stats = construir_category_stats(saneado, categoria["categoria_referencia"])
print(f"category_stats: {len(category_stats):,} filas (categoría × nutriente)")
print(f"Categorías de referencia distintas con al menos un nutriente válido: {category_stats['categoria_referencia'].nunique():,}")

# Cuántos percentiles válidos tiene cada producto, de los 8 posibles.
n_percentiles_validos = percentiles.notna().sum(axis=1)
n_percentiles_validos.value_counts().sort_index().rename("n_productos").to_frame()

category_stats: 1,608 filas (categoría × nutriente)
Categorías de referencia distintas con al menos un nutriente válido: 201


,n_productos
0,9792
1,19
2,19
3,16
4,87
5,95
6,205
7,531
8,6087


In [9]:
# Ejemplo de category_stats para una categoría concreta y conocida, para
# verificar a ojo que los números son razonables (auditoría manual mínima).
category_stats[category_stats["categoria_referencia"] == "en:yogurts"]

,categoria_referencia,nutriente,n,media,mediana,desviacion_estandar,minimo,maximo
193,en:yogurts,energy-kcal_100g,91,83.725917,79.155556,33.018301,27.727273,226.0
394,en:yogurts,fat_100g,91,2.119972,1.8,2.249045,0.0,18.0
595,en:yogurts,saturated-fat_100g,90,1.190655,1.1,0.959766,0.0,3.7
796,en:yogurts,carbohydrates_100g,90,11.924454,11.627778,5.427521,1.6,32.0
997,en:yogurts,sugars_100g,88,10.121962,9.9,5.016383,1.636364,31.0
1198,en:yogurts,proteins_100g,91,4.382554,3.6,2.74316,0.9,17.6
1399,en:yogurts,salt_100g,82,0.138632,0.120875,0.138827,0.0,1.25
1600,en:yogurts,fiber_100g,82,0.395278,0.1,0.651009,0.0,3.0


### Hallazgo 3 — la cobertura de percentiles ronda 39-41% por nutriente, y 6.055 productos (35,9%) tienen los 8 percentiles completos

Verificado el 2026-09-20: cada nutriente del core8 termina con entre 38,9% (fibra) y 41,5% (grasa) de percentiles válidos — más bajo que el 48,1% de productos con categoría de referencia resuelta, porque además hace falta el dato saneado del nutriente y el mínimo de 5 pares. La caída es moderada (unos 7-9 puntos porcentuales), no catastrófica.

`category_stats` tiene 1.608 filas para 201 categorías de referencia distintas — bastante más categorías que las top-20 que se listaron en el EDA, lo cual confirma que la regla de retroceso está usando un espectro amplio de categorías, no solo unas pocas genéricas.

La distribución de percentiles por producto es marcadamente bimodal: 9.793 productos (58,1%) no tienen ningún percentil válido, y 6.055 (35,9%) tienen los 8 completos; los valores intermedios (1 a 7 percentiles) juntos suman menos del 6% del universo. Esto tiene sentido: si un producto tiene categoría de referencia y su matriz nutricional está razonablemente completa (que suele ir junta, ver EDA sección 4), es probable que tenga casi todos los nutrientes o ninguno.


## 4. D2 — NOVA combinado con `additives_n` (decisión A17)

`src/nutrimatch/engine/processing_score.py`. **La fórmula exacta es una propuesta de esta transformación, no estaba cerrada en A17** (que solo definía *qué* combinar). Se propone: cada grupo NOVA ocupa una banda de 25 puntos (NOVA=1 → 75-100, ..., NOVA=4 → 0-25), preservando que NOVA siempre pese más que `additives_n` (ningún producto de mejor NOVA puede terminar por debajo de uno de peor NOVA). Dentro de su banda, `additives_n` empuja hacia el piso hasta un tope calibrado como el percentil 95 de `additives_n` entre los productos NOVA=4 del propio snapshot.

**Pendiente de tu aprobación explícita**, ver hallazgo 4.


In [10]:
tope_aditivos = calibrar_tope_aditivos(df["nova_group"], df["additives_n"], percentil=0.95)
d2 = calcular_d2(df["nova_group"], df["additives_n"], tope_aditivos=tope_aditivos)

print(f"Tope de aditivos calibrado (percentil 95 de additives_n en NOVA=4): {tope_aditivos}")
print(f"Productos con D2 calculable (tienen NOVA): {d2.notna().sum():,} ({d2.notna().sum()*100/N:.1f}%)\n")

resumen_d2 = pd.DataFrame({"nova_group": df["nova_group"], "d2": d2})
resumen_d2 = resumen_d2[resumen_d2["nova_group"].notna() & (resumen_d2["nova_group"].str.strip() != "")]
resumen_d2.groupby("nova_group")["d2"].agg(n="count", media="mean", minimo="min", maximo="max").sort_index()

Tope de aditivos calibrado (percentil 95 de additives_n en NOVA=4): 9.0
Productos con D2 calculable (tienen NOVA): 6,781 (40.2%)



,n,media,minimo,maximo
nova_group,,,,
1,804,99.768519,91.666667,100.0
2,189,74.764844,69.444444,75.0
3,1078,48.626572,30.555556,50.0
4,4710,15.202878,0.0,25.0


In [11]:
fig = px.histogram(
    resumen_d2, x="d2", color="nova_group", nbins=40,
    title="Distribución de D2 por grupo NOVA (bandas de 25 puntos, sin solaparse)",
    labels={"d2": "Subpuntaje D2 (0-100)", "count": "Nº de productos", "nova_group": "Grupo NOVA"},
    category_orders={"nova_group": ["1", "2", "3", "4"]},
)
fig.add_vline(x=25, line_dash="dot", line_color="gray")
fig.add_vline(x=50, line_dash="dot", line_color="gray")
fig.add_vline(x=75, line_dash="dot", line_color="gray")
fig.show()

### Hallazgo 4 — las bandas quedan bien separadas y usadas en su rango casi completo; el tope calibrado es 9 aditivos

Verificado el 2026-09-20: con el tope calibrado en 9,0 (p95 de `additives_n` dentro de NOVA=4), las cuatro bandas quedan claramente separadas y cada una se usa en casi todo su rango: NOVA=1 va de 91,7 a 100 (media 99,8 — casi todos los productos NOVA=1 tienen 0 aditivos, como se esperaba), NOVA=2 de 69,4 a 75, NOVA=3 de 30,6 a 50, y NOVA=4 de 0 a 25 (media 15,2, el grupo con más dispersión interna, coherente con ser el que tiene más variedad de conteos de aditivos según el EDA). El grupo NOVA=4 (4.710 productos, el 69,5% de los que tienen NOVA, igual que en el EDA) deja de ser un bloque plano de "100% grupo 4" y pasa a repartirse en 25 puntos de rango según cuántos aditivos declara.

**Esta fue una propuesta con tres parámetros abiertos** — (1) el ancho de banda fijo de 25 puntos por grupo, (2) el percentil de calibración (0,95), (3) tratar `additives_n` ausente como 0 aditivos — **confirmada por Paola sin cambios el 2026-09-20 (decisión A23, ver síntesis final).**


## 5. Ensamblado de `matriz_nut_100g`, trazabilidad y guardado

Se combinan todas las piezas en una sola tabla por producto (`code`), con `snapshot_id` para trazabilidad, incluyendo el flag `universo_puntuable` (decisión A22, confirmada 2026-09-20: ≥4 de 8 percentiles válidos + D2 calculable). No se calcula aquí el score final ponderado — eso depende de los pesos del usuario (`cov`, paso 7) —, pero sí se deja tanto el flag binario como el conteo objetivo de percentiles válidos por producto para que el paso 7 los use.


In [12]:
matriz_nut_100g = pd.DataFrame({"code": df["code"], "snapshot_id": SNAPSHOT_ID})
matriz_nut_100g = matriz_nut_100g.join(saneado)
matriz_nut_100g["flag_suma_macros_excede_100"] = flag_macros
matriz_nut_100g["categoria_referencia"] = categoria["categoria_referencia"]
matriz_nut_100g["nivel_retroceso"] = categoria["nivel_retroceso"]
matriz_nut_100g = matriz_nut_100g.join(percentiles)
matriz_nut_100g["n_percentiles_validos"] = n_percentiles_validos
matriz_nut_100g["nova_group"] = df["nova_group"]
matriz_nut_100g["additives_n"] = df["additives_n"]
matriz_nut_100g["d2"] = d2

# Decisión A22 (confirmada 2026-09-20): universo puntuable = ≥4 de 8 percentiles
# válidos (al menos la mitad del núcleo nutricional) Y D2 calculable (tiene NOVA).
UMBRAL_PERCENTILES_PUNTUABLE = 4
matriz_nut_100g["universo_puntuable"] = (
    (matriz_nut_100g["n_percentiles_validos"] >= UMBRAL_PERCENTILES_PUNTUABLE) & matriz_nut_100g["d2"].notna()
)

category_stats["snapshot_id"] = SNAPSHOT_ID

print(f"matriz_nut_100g: {matriz_nut_100g.shape[0]:,} filas × {matriz_nut_100g.shape[1]} columnas")
print(f"category_stats:  {category_stats.shape[0]:,} filas × {category_stats.shape[1]} columnas")
assert matriz_nut_100g["code"].is_unique, "code debe ser único: ya se verificó en el EDA (0 duplicados)"
matriz_nut_100g.head(3)

matriz_nut_100g: 16,851 filas × 43 columnas
category_stats:  1,608 filas × 9 columnas


,code,snapshot_id,energy-kcal_100g_bruto,energy-kcal_100g_flag_fuera_de_rango,energy-kcal_100g_saneado,fat_100g_bruto,fat_100g_flag_fuera_de_rango,fat_100g_saneado,saturated-fat_100g_bruto,saturated-fat_100g_flag_fuera_de_rango,...,percentil_carbohydrates_100g,percentil_sugars_100g,percentil_proteins_100g,percentil_salt_100g,percentil_fiber_100g,n_percentiles_validos,nova_group,additives_n,d2,universo_puntuable
0,00000140,off_csv_20260919,<NA>,False,<NA>,<NA>,False,<NA>,<NA>,False,...,<NA>,<NA>,<NA>,<NA>,<NA>,0,4,0,25.0,False
1,00000154,off_csv_20260919,<NA>,False,<NA>,<NA>,False,<NA>,<NA>,False,...,<NA>,<NA>,<NA>,<NA>,<NA>,0,4,1,22.222222,False
2,00000285,off_csv_20260919,<NA>,False,<NA>,<NA>,False,<NA>,<NA>,False,...,<NA>,<NA>,<NA>,<NA>,<NA>,0,NaN,NaN,<NA>,False


In [13]:
ruta_matriz = REPO_ROOT / "datos" / "procesados" / "matriz_nut_100g_20260919.parquet"
ruta_category_stats = REPO_ROOT / "datos" / "procesados" / "category_stats_20260919.parquet"

matriz_nut_100g.to_parquet(ruta_matriz, index=False)
category_stats.to_parquet(ruta_category_stats, index=False)

# Verificación de reproducibilidad: recargar desde disco y confirmar shapes.
verificacion_matriz = pd.read_parquet(ruta_matriz)
verificacion_stats = pd.read_parquet(ruta_category_stats)
assert verificacion_matriz.shape == matriz_nut_100g.shape
assert verificacion_stats.shape == category_stats.shape

print(f"Guardado: {ruta_matriz.relative_to(REPO_ROOT)} ({ruta_matriz.stat().st_size / 1024:.0f} KB)")
print(f"Guardado: {ruta_category_stats.relative_to(REPO_ROOT)} ({ruta_category_stats.stat().st_size / 1024:.0f} KB)")
print("Verificación de reproducibilidad: shapes coinciden tras recargar desde disco. OK.")

Guardado: datos/procesados/matriz_nut_100g_20260919.parquet (930 KB)
Guardado: datos/procesados/category_stats_20260919.parquet (55 KB)
Verificación de reproducibilidad: shapes coinciden tras recargar desde disco. OK.


## 6. Verificación cruzada contra la estimación del EDA (universo A19)

La sección 8 del EDA estimó el universo puntuable de forma naive (core5 + tiene categoría + tiene NOVA, sin resolver categoría de referencia ni exigir peers mínimos) en 5.856 productos (34,8%). Con el cálculo preciso de esta transformación — que sí exige categoría de referencia resuelta y percentiles con peers suficientes —, se puede recalcular esa misma cifra de forma exacta.


In [14]:
tiene_d2 = matriz_nut_100g["d2"].notna()
for minimo in (1, 4, 8):
    k = ((matriz_nut_100g["n_percentiles_validos"] >= minimo) & tiene_d2).sum()
    marca = "  ← umbral elegido (A22)" if minimo == UMBRAL_PERCENTILES_PUNTUABLE else ""
    print(f"≥{minimo} percentiles válidos de 8 AND D2 calculable: {k:,} ({k*100/N:.1f}%){marca}")
print("\nEstimación naive del EDA (sección 8, sin resolver categoría de referencia): 5.856 (34,8%)")
print(f"\nuniverso_puntuable final (columna guardada en matriz_nut_100g): "
      f"{matriz_nut_100g['universo_puntuable'].sum():,} ({matriz_nut_100g['universo_puntuable'].mean()*100:.1f}%)")

≥1 percentiles válidos de 8 AND D2 calculable: 5,888 (34.9%)
≥4 percentiles válidos de 8 AND D2 calculable: 5,864 (34.8%)  ← umbral elegido (A22)
≥8 percentiles válidos de 8 AND D2 calculable: 5,341 (31.7%)

Estimación naive del EDA (sección 8, sin resolver categoría de referencia): 5.856 (34,8%)

universo_puntuable final (columna guardada en matriz_nut_100g): 5,864 (34.8%)


### Hallazgo 5 — la cifra precisa (con ≥1 percentil) coincide casi exactamente con la naive del EDA: 34,9% vs 34,8%

Verificado el 2026-09-20 (tras la corrección de escala de sal, decisión A21): exigiendo al menos 1 de 8 percentiles válidos y D2 calculable, el universo preciso da 5.888 productos (34,9%), prácticamente idéntico al 34,8% naive del EDA. Con el umbral más estricto de exigir los 8 percentiles completos, baja a 5.341 (31,7%) — una diferencia de 3,2 puntos porcentuales, que es el costo real de exigir la matriz nutricional íntegra en vez de solo parcial.

Esta coincidencia es una señal de consistencia entre el EDA y esta transformación: los métodos más simples y los más rigurosos convergen en una cifra similar, lo cual da confianza en que el 34,8% de A19 (universo puntuable, decisión ya aprobada) es una estimación razonable — el paso 7 puede afinar el umbral exacto (¿1 percentil basta, o se requieren más?) sin que eso cambie drásticamente el tamaño del universo puntuable.

---

## Síntesis y decisiones pendientes de tu aprobación

**Construido en esta transformación** (todo en `src/nutrimatch/engine/`, con 17 pruebas unitarias en `tests/`):

| Pieza | Módulo | Resultado verificado |
|---|---|---|
| Saneamiento core8 | `sanitize.py` | 3-58 productos marcados por nutriente; hallazgo nuevo de error sistemático de escala en sal |
| Categoría de referencia | `reference_category.py` | 48,1% del universo resuelto (8.101 productos) |
| Percentiles de D1 | `nutrition_percentile.py` | 38,9%-41,5% de cobertura por nutriente |
| `category_stats` | `nutrition_percentile.py` | 1.608 filas, 201 categorías |
| D2 (NOVA + additives_n) | `processing_score.py` | Bandas separadas, tope calibrado en 9,0 |

**Archivos guardados:** `datos/procesados/matriz_nut_100g_20260919.parquet`, `datos/procesados/category_stats_20260919.parquet`.

**Cuatro decisiones nuevas, confirmadas por Paola el 2026-09-20** (se registran en `AGENTS.md` como A21-A24):

1. **A21 — Corrección de escala de sal:** cuando `salt_100g` está fuera de rango y su razón con `sodium_100g` es ≈2,5 (fórmula química estándar) y el valor reescalado (÷1000) cae en rango válido, se corrige y se usa en percentiles. 55 de 56 casos calificaron; el caso atípico (razón ≈2500) sigue excluido. Trazable vía `salt_100g_flag_correccion_escala_aplicada`.
2. **A22 — Umbral del universo puntuable:** ≥4 de 8 percentiles válidos (al menos la mitad del núcleo) Y D2 calculable. Columna `universo_puntuable` guardada en `matriz_nut_100g`. Da 5.864 productos (34,8%), consistente con la estimación naive de A19.
3. **A23 — Fórmula de D2 confirmada tal cual se propuso:** bandas de 25 puntos por grupo NOVA, ajuste por `additives_n` con tope calibrado en el percentil 95 de `additives_n` dentro de NOVA=4 (da 9,0 en este snapshot), y `additives_n` ausente tratado como 0.
4. **A24 — `MINIMO_PEERS_PERCENTIL = 5` confirmado** tal cual se propuso, sin cambios.

Con esto, la transformación queda cerrada y lista para pasar al paso 7 (modelo de recomendación): aplicar el signo de D1 según el objetivo del usuario, D3 (preferencias), la regla de cobertura `cov` con pesos del usuario, y los filtros duros de alergia y dieta (A15).
